# Summary — Note Summarisation

Runs summarisation across the vault using the configured backend and compares
results against the ground truth in `data/ground_truth.yaml`.

Unlike NER and tags, summaries are compared by reading — there is no exact-match metric.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_summary

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
)
print(f'{len(notes)} notes found')

20 notes found


## Run summarisation

In [2]:
import time

backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})

    row = {
        'file': name,
        'lang': gt_entry.get('lang', '?'),
        'ground_truth': gt_entry.get('summary', ''),
    }

    t0 = time.perf_counter()
    try:
        sentences = run_summary(backend, post.content, config)
        row[backend] = ' '.join(sentences)
    except Exception as e:
        row[backend] = f'ERROR: {e}'
    row['elapsed_s'] = time.perf_counter() - t0

    results.append(row)
    print(f'  {name}  ({row["elapsed_s"]:.1f}s)')

print(f'\nDone — {len(results)} notes processed')

  long_ai_gezondheidszorg_oncologie_onderzoek_nl.md  (8.9s)
  long_ai_healthcare_oncology_research_en.md  (4.7s)
  long_interview_climate_scientist_sea_level_en.md  (6.2s)
  long_interview_klimaatwetenschapper_zeespiegel_nl.md  (7.7s)
  long_opinie_eu_ai_act_digitaal_beleid_nl.md  (6.7s)
  long_opinion_eu_ai_act_digital_policy_en.md  (5.2s)
  medium_climate_blog_paris_agreement_en.md  (3.4s)
  medium_geschiedenis_gouden_eeuw_nl.md  (6.2s)
  medium_history_dutch_golden_age_en.md  (3.9s)
  medium_klimaat_blog_parijs_akkoord_nl.md  (5.0s)
  medium_reisgids_amsterdam_nl.md  (3.7s)
  medium_travel_guide_amsterdam_en.md  (4.9s)
  short_recensie_samsung_galaxy_s25_ultra_nl.md  (2.8s)
  short_recept_stamppot_nl.md  (3.8s)
  short_recipe_dutch_stamppot_en.md  (3.5s)
  short_review_samsung_galaxy_s25_ultra_en.md  (3.7s)
  short_sport_nieuws_ajax_champions_league_nl.md  (3.4s)
  short_sports_news_ajax_champions_league_en.md  (2.5s)
  short_tech_news_openai_announcement_en.md  (2.9s)
  short_tech_

In [ ]:
import json

out_dir = Path('../results/summary')
out_dir.mkdir(parents=True, exist_ok=True)

for r in results:
    stem = Path(r['file']).stem
    payload = r[backend]
    is_error = isinstance(payload, str) and payload.startswith('ERROR:')
    data = {
        'file': r['file'],
        'backend': backend,
        'model': config[backend]['model'],
        'elapsed_s': round(r['elapsed_s'], 3),
        'ground_truth': r['ground_truth'],
        'summary': payload if not is_error else '',
        'error': payload if is_error else None,
    }
    (out_dir / f'{stem}.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

print(f'Saved {len(results)} files to {out_dir}')

## Comparison table

In [4]:
pd.set_option('display.max_colwidth', 120)
df = pd.DataFrame([{
    'file': r['file'],
    'lang': r['lang'],
    'ground_truth': r['ground_truth'],
    backend: r[backend],
} for r in results])
df

,file,lang,ground_truth,ollama
0,long_ai_gezondheidszorg_oncologie_onderzoek_nl.md,nl,"A clinical trial at Amsterdam UMC involving 2,847 patients found that AI-assisted pathology (PathAI Colorectal Class...",Een gerandomiseerde gecontroleerde studie van twee jaar uitgevoerd in Amsterdam UMC heeft aangetoond dat kunstmatige...
1,long_ai_healthcare_oncology_research_en.md,en,"A clinical trial at Amsterdam UMC involving 2,847 patients found that AI-assisted pathology (PathAI Colorectal Class...",A two-year randomised controlled trial at Amsterdam UMC demonstrated that AI-assisted pathology can significantly re...
2,long_interview_climate_scientist_sea_level_en.md,en,"An interview with Dr. Maria van den Berg, professor of coastal hydrology at TU Delft and senior fellow at Deltares, ...","The Netherlands is better prepared than most countries for sea level rise, but not fully prepared for the current sc..."
3,long_interview_klimaatwetenschapper_zeespiegel_nl.md,nl,"An interview with Dr. Maria van den Berg, professor of coastal hydrology at TU Delft and senior fellow at Deltares, ...",Nederland is beter voorbereid dan vrijwel elk ander land op de zeespiegelstijging tegen 2100 De Deltawerken bescherm...
4,long_opinie_eu_ai_act_digitaal_beleid_nl.md,nl,"An opinion piece arguing that while the EU AI Act is a genuine regulatory achievement, its implementation is undermi...",De EU AI Act is een wetgevende architectuur die systemen met 'onaanvaardbaar risico' verbiedt en 'hoog-risico'-toepa...
5,long_opinion_eu_ai_act_digital_policy_en.md,en,"An opinion piece arguing that while the EU AI Act is a genuine regulatory achievement, its implementation is undermi...","The EU AI Act introduces a tiered risk framework for AI systems, with bans on unacceptable risk applications and con..."
6,medium_climate_blog_paris_agreement_en.md,en,"Ten years after the Paris Agreement was signed in Le Bourget, France, global CO₂ emissions have reached record level...","Global average temperatures are 1.45°C above pre-industrial levels, close to the 2°C threshold. Fossil fuel consumpt..."
7,medium_geschiedenis_gouden_eeuw_nl.md,nl,"An encyclopaedic overview of the Dutch Golden Age (1588-1702), covering the founding of the VOC in Amsterdam in 1602...","The Dutch Golden Age (1588-1702) was a period of exceptional achievements in trade, science, military power and art...."
8,medium_history_dutch_golden_age_en.md,en,"An encyclopaedic overview of the Dutch Golden Age (1588-1702), covering the founding of the VOC in Amsterdam in 1602...",The Dutch Golden Age was a period of extraordinary achievement from 1588 to 1702. The VOC established its Asian head...
9,medium_klimaat_blog_parijs_akkoord_nl.md,nl,"Ten years after the Paris Agreement was signed in Le Bourget, France, global CO₂ emissions have reached record level...","Tien jaar na Parijs: het beeld is ontnuchterend De mondiale CO2-uitstoot bereikte in 2023 een record van 37,4 gigato..."


## Inspect a single note

Change `INSPECT` to read one note's summaries in full.

In [5]:
INSPECT = Path(notes[0]).name

row = next((r for r in results if r['file'] == INSPECT), None)
if row:
    print(f'=== {INSPECT} ===\n')
    print('GROUND TRUTH:')
    print(row['ground_truth'])
    print(f'\n{backend.upper()} ({config[backend]["model"]}):')
    print(row[backend])

=== long_ai_gezondheidszorg_oncologie_onderzoek_nl.md ===

GROUND TRUTH:
A clinical trial at Amsterdam UMC involving 2,847 patients found that AI-assisted pathology (PathAI Colorectal Classifier v3.1) improved early-stage colorectal cancer diagnostic accuracy from 91.7% to 96.3% and reduced reporting time by 38.7%. The trial, led by Dr. Jan de Vries and Prof. Maria Santos, was published in The Lancet Digital Health and received CE marking under EU MDR. Philips Healthcare and PathAI announced a commercial partnership to roll out the technology to 35 Dutch hospitals by Q1 2026.

OLLAMA (llama3.2):
Een gerandomiseerde gecontroleerde studie van twee jaar uitgevoerd in Amsterdam UMC heeft aangetoond dat kunstmatige intelligentie-ondersteunde pathologie diagnostische fouten bij darmkanker in een vroeg stadium aanzienlijk kan verminderen, terwijl de rapportagetijd met 38% wordt verkort. De bevindingen vertegenwoordenen een van de grootste prospectieve studies naar AI-ondersteunde oncologische

## (Optional) Save results to frontmatter

Uncomment to write results under `notemine.summary.<backend>`.

In [6]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     summary_data = notemine.get('summary', {})
#     if not row.get(backend, '').startswith('ERROR'):
#         summary_data[backend] = row[backend]
#     notemine['summary'] = summary_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')